In [ ]:
# To avoid learning fail (log or denominator)
s = 1e-5

import pandas as pd
from sklearn import metrics
import tensorflow as tf
import keras
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from keras import optimizers
from tensorflow.keras.layers import Dense
from keras.layers import BatchNormalization
from keras.layers import Activation
from keras import optimizers
from keras import initializers

def concath(df_X, df_y):
    df = pd.concat([df_X, df_y])
    return df


################################ MSE ################################
def MSE(y_true, y_pred):
    return tf.reduce_mean(tf.math.square(y_true-y_pred))

################################ BCE ################################
def BCE(y_true, y_pred):
    return -tf.reduce_mean(y_true*tf.math.log(y_pred+s)+(1-y_true)*tf.math.log(1-y_pred+s))

################################ WBCE ################################
def WBCE(y_true, y_pred):
    N = batch    # batch_size
    y1 = tf.reduce_sum(y_true)
    y0 = N-y1
    w1 = y0/N #N/y1
    w0 = y1/N #N/y0
    return -tf.reduce_mean(w1*y_true*tf.math.log(y_pred+s)+w0*(1-y_true)*tf.math.log(1-y_pred+s))

################################ TN/FP/FN/TP ################################
def confusion_matrix(y_true, y_pred):
    N = batch    # batch_size
    y1 = tf.reduce_sum(y_true)
    y0 = N-y1
    TN = N-tf.reduce_sum(y_true)-tf.reduce_sum(y_pred)+tf.reduce_sum(y_true*y_pred)
    FP = tf.reduce_sum(y_pred)-tf.reduce_sum(y_true*y_pred)
    FN = tf.reduce_sum(y_true)-tf.reduce_sum(y_true*y_pred)
    TP = tf.reduce_sum(y_true*y_pred)
    return N, y1, y0, TN, FP, FN, TP

################################ make_lists ################################
def make_lists():
    list_acc = []
    list_f1 = []
    list_gmean = []
    list_bacc = []
    list_pre = []
    list_rec = []
    list_spe = []
    return list_acc, list_f1, list_gmean, list_bacc, list_pre, list_rec, list_spe
    
############################### Results ###############################
def get_results(y_true, y_pred):
    TN = metrics.confusion_matrix(y_true, y_pred)[0,0]
    FP = metrics.confusion_matrix(y_true, y_pred)[0,1]
    FN = metrics.confusion_matrix(y_true, y_pred)[1,0]
    TP = metrics.confusion_matrix(y_true, y_pred)[1,1]
    acc = np.round((TP+TN)/(TP+TN+FP+FN),4)
    if TP+FP == 0:
        pre = 0
    else:
        pre = np.round(TP/(TP+FP),4)
    rec = np.round(TP/(TP+FN),4)
    spe = np.round(TN/(TN+FP),4)
    f1 = np.round(TP/(TP + 0.5*(FP+FN)),4)
    f05 = np.round(TP/(TP + 0.8*FP + 0.2*FN),4)
    f2 = np.round(TP/(TP + 0.2*FP + 0.8*FN),4)
    gmean = np.round(((TP/(TP+FN)) * (TN/(TN+FP)))**0.5,4)
    bacc = np.round(0.5*(TP/(TP+FN) + TN/(TN+FP)),4)
    
    list_acc.append(acc)
    list_f1.append(f1)
    list_gmean.append(gmean)
    list_bacc.append(bacc)
    list_pre.append(pre)
    list_rec.append(rec)
    list_spe.append(spe)
    
#     print("Acc:", acc)
#     print("F1:", f1)
#     print("G_mean:", gmean)
#     print("B_Acc:", bacc)
#     print("PRE:", pre)
#     print("REC:", rec)
#     print("SPE:", spe)
    
#     return acc, pre, rec, spe, f1, f05, f2, gmean, bacc

################################ SPL ################################
def splitter(y_pred):
    return (0.5)**2-(y_pred-0.5)**2

# =================================== Accuracy =================================== #
################################ Pure_Accu ################################
def Pure_Accu(y_true, y_pred):
    N, y1, y0, TN, FP, FN, TP = confusion_matrix(y_true, y_pred)
    accu = (TP+TN)/N
    return 1-accu

################################ Any_Accu ################################
def Any_Accu(y_true, y_pred):
    y_pred = 1/(1+tf.math.exp(-L*(y_pred-0.5)))
    N, y1, y0, TN, FP, FN, TP = confusion_matrix(y_true, y_pred)
    accu = (TP+TN)/N
    return 1-accu

# ################################ BCEAL ################################
# def BCEAL(y_true, y_pred):
#     BCEloss = BCE(y_true, y_pred)
#     N, y1, y0, TN, FP, FN, TP = confusion_matrix(y_true, y_pred)
#     accu = (TP+TN)/N
#     return (1-r)*BCEloss+(r)*(1-accu)

################################ WBCEAL ################################
def WBCEAL(y_true, y_pred):
    WBCEloss = WBCE(y_true, y_pred)
    N, y1, y0, TN, FP, FN, TP = confusion_matrix(y_true, y_pred)
    accu = (TP+TN)/N
    return (1-r)*WBCEloss+(r)*(1-accu)

################################ SPLAL ################################
def SPLAL(y_true, y_pred):
    SPL = splitter(y_pred)
    N, y1, y0, TN, FP, FN, TP = confusion_matrix(y_true, y_pred)
    accu = (TP+TN)/N
    return (1-w)*SPL+(w)*(1-accu)

# =================================== Fbeta =================================== #
################################ Pure_Fbeta ################################
def Pure_Fbeta(y_true, y_pred):
    b = 1 
    N, y1, y0, TN, FP, FN, TP = confusion_matrix(y_true, y_pred)
    F_beta = ((1+b**2)*TP) / ((b**2)*y1 + tf.reduce_sum(y_pred)+s)  # (1+b**2)TP/((1+b**2)TP+FP+b**2FN)
    return 1-F_beta

################################ Any_Fbeta ################################
def Any_Fbeta(y_true, y_pred):
    b = 1 
    y_pred = 1/(1+tf.math.exp(-L*(y_pred-0.5)))
    N, y1, y0, TN, FP, FN, TP = confusion_matrix(y_true, y_pred)
    F_beta = ((1+b**2)*TP) / ((b**2)*y1 + tf.reduce_sum(y_pred)+s)  # (1+b**2)TP/((1+b**2)TP+FP+b**2FN)
    return 1-F_beta

# ################################ BCEFL ################################
# def BCEFL(y_true, y_pred):
#     b = 1
#     BCEloss = BCE(y_true, y_pred)
#     N, y1, y0, TN, FP, FN, TP = confusion_matrix(y_true, y_pred)
#     F_beta = ((1+b**2)*TP) / ((b**2)*y1 + tf.reduce_sum(y_pred)+s)  # (1+b**2)TP/((1+b**2)TP+FP+b**2FN)
#     return (1-r)*BCEloss+(r)*(1-F_beta)

################################ WBCEFL ################################
def WBCEFL(y_true, y_pred):
    b = 1
    WBCEloss = WBCE(y_true, y_pred)
    N, y1, y0, TN, FP, FN, TP = confusion_matrix(y_true, y_pred)
    F_beta = ((1+b**2)*TP) / ((b**2)*y1 + tf.reduce_sum(y_pred)+s)  # (1+b**2)TP/((1+b**2)TP+FP+b**2FN)
    return (1-r)*WBCEloss+(r)*(1-F_beta)

################################ SPLFL ################################
def SPLFL(y_true, y_pred):
    b = 1
    SPL = splitter(y_pred)
    N, y1, y0, TN, FP, FN, TP = confusion_matrix(y_true, y_pred)
    F_beta = ((1+b**2)*TP) / ((b**2)*y1 + tf.reduce_sum(y_pred)+s)  # (1+b**2)TP/((1+b**2)TP+FP+b**2FN)
    return (1-w)*SPL+(w)*(1-F_beta)

# =================================== Gmean =================================== #
################################ Pure_Gmean ################################
def Pure_Gmean(y_true, y_pred):
    N, y1, y0, TN, FP, FN, TP = confusion_matrix(y_true, y_pred)
    sur_gmean = (TP*TN)/(y1*y0+s)
    return 1-sur_gmean

################################ Any_Gmean ################################
def Any_Gmean(y_true, y_pred):
    y_pred = 1/(1+tf.math.exp(-L*(y_pred-0.5)))
    N, y1, y0, TN, FP, FN, TP = confusion_matrix(y_true, y_pred)
    sur_gmean = (TP*TN)/(y1*y0+s)
    return 1-sur_gmean

# ################################ BCEGL ################################
# def BCEGL(y_true, y_pred):
#     BCEloss = BCE(y_true, y_pred)
#     N, y1, y0, TN, FP, FN, TP = confusion_matrix(y_true, y_pred)
#     sur_gmean = (TP*TN)/(y1*y0+s)
#     return (1-r)*BCEloss+(r)*(1-sur_gmean)

################################ WBCEGL ################################
def WBCEGL(y_true, y_pred):
    WBCEloss = WBCE(y_true, y_pred)
    N, y1, y0, TN, FP, FN, TP = confusion_matrix(y_true, y_pred)
    sur_gmean = (TP*TN)/(y1*y0+s)
    return (1-r)*WBCEloss+(r)*(1-sur_gmean)

################################ SPLGL ################################
def SPLGL(y_true, y_pred):
    SPL = splitter(y_pred)
    N, y1, y0, TN, FP, FN, TP = confusion_matrix(y_true, y_pred)
    sur_gmean = (TP*TN)/(y1*y0+s)
    return (1-w)*SPL+(w)*(1-sur_gmean)

# =================================== BAccu =================================== #
################################ Pure_BAccu ################################
def Pure_BAccu(y_true, y_pred):
    N, y1, y0, TN, FP, FN, TP = confusion_matrix(y_true, y_pred)
    baccu = (y0*TP+y1*TN) / (2*y1*y0+s)
    return 1-baccu

################################ Any_BAccu ################################
def Any_BAccu(y_true, y_pred):
    y_pred = 1/(1+tf.math.exp(-L*(y_pred-0.5)))
    N, y1, y0, TN, FP, FN, TP = confusion_matrix(y_true, y_pred)
    baccu = (y0*TP+y1*TN) / (2*y1*y0+s)
    return 1-baccu

# ################################ BCEBL ################################
# def BCEBL(y_true, y_pred):
#     BCEloss = BCE(y_true, y_pred)
#     N, y1, y0, TN, FP, FN, TP = confusion_matrix(y_true, y_pred)
#     baccu = (y0*TP+y1*TN) / (2*y1*y0+s)
#     return (1-r)*BCEloss+(r)*(1-baccu)

################################ WBCEBL ################################
def WBCEBL(y_true, y_pred):
    WBCEloss = WBCE(y_true, y_pred)
    N, y1, y0, TN, FP, FN, TP = confusion_matrix(y_true, y_pred)
    baccu = (y0*TP+y1*TN) / (2*y1*y0+s)
    return (1-r)*WBCEloss+(r)*(1-baccu)

################################ SPLBL ################################
def SPLBL(y_true, y_pred):
    SPL = splitter(y_pred)
    N, y1, y0, TN, FP, FN, TP = confusion_matrix(y_true, y_pred)
    baccu = (y0*TP+y1*TN) / (2*y1*y0+s)
    return (1-w)*SPL+(w)*(1-baccu)

# Diabetes Prediction Data (8d / 100000)

In [ ]:
# class '0' = normal, class '1' = anomaly
diab_df = pd.read_csv(r'/afs/crc.nd.edu/user/d/dhan6/3. Loss Function/diabetes_prediction_dataset.csv')
diab_df.shape

In [ ]:
diab_df.isnull().sum()

In [ ]:
diab_df.head()

In [ ]:
# Female = 0, Male = 1, other = 2
gen_encoded, gen_class = pd.factorize(diab_df['gender'])
print(gen_class)
gen_encoded

In [ ]:
# Female = 0, Male = 1, other = 2
pd.Series(gen_encoded).value_counts()

In [ ]:
diab_df['gender'] = gen_encoded
diab_df

In [ ]:
# never = 0, Info = 1, current = 2, former=3, ever=4, not current=5
smo_encoded, smo_class = pd.factorize(diab_df['smoking_history'])
print(smo_class)
smo_encoded

In [ ]:
# never = 0, Info = 1, current = 2, former=3, ever=4, not current=5
pd.Series(smo_encoded).value_counts()

In [ ]:
diab_df['smoking_history'] = smo_encoded
diab_df

In [ ]:
diab_df.describe()

In [ ]:
# Standardization
diab_df.iloc[:,:-1] = (diab_df.iloc[:,:-1] - diab_df.iloc[:,:-1].mean())/diab_df.iloc[:,:-1].std()

diab_df

In [ ]:
diab_df['diabetes'].value_counts()

In [ ]:
res = pd.DataFrame({'MSE':[0, 0, 0, 0, 0, 0, 0]}, index = ['Acc','F1','G_Mean','B_Acc','Pre','Rec','Spe'])
res

In [ ]:
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state = 2)

hidden_node = 2
activation = 'sigmoid'  
kernel_initializer=tf.keras.initializers.he_normal(seed=100)
epochs = 50
threshold = 0.5
L = 30
learning_rate = 0.01

X = diab_df.iloc[:, :-1]
y = diab_df.iloc[:, -1]

In [ ]:
def create_model():
    model = Sequential()
    model.add(Dense(hidden_node, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
    model.add(BatchNormalization())
    model.add(Activation(activation))
    model.add(Dense(1, activation='sigmoid'))
    return model

# MSE

In [ ]:
mse_acc = []
mse_f1 = []
mse_gmean = []
mse_bacc = []
mse_pre = []
mse_rec = []
mse_spe = []

for i in range(5):
    print('#'*50,'{0}th repeat'.format(i+1),'#'*50)
    list_acc, list_f1, list_gmean, list_bacc, list_pre, list_rec, list_spe = make_lists()

    n_iter=0
    ###################### MLP (sigmoid // MSE) ##############################
    for train_index, test_index in skf.split(X, y):
        n_iter += 1
        X_train = X.iloc[train_index]
        y_train= y.iloc[train_index]
        X_test = X.iloc[test_index]
        y_test= y.iloc[test_index]
        X_train = np.array(X_train)
        y_train = np.array(y_train)
        y_train = y_train.astype(float)
        X_test = np.array(X_test)
        y_test = np.array(y_test)
        y_test = y_test.astype(float)
        batch = int(X_train.shape[0] * 0.05)

        model = create_model()
        opt = tf.keras.optimizers.Adam(learning_rate = 0.01)
        model.compile(optimizer=opt, loss=MSE, metrics=['accuracy'])
        history = model.fit(X_train, y_train, validation_data=(X_test, y_test), verbose=0, epochs=epochs, batch_size = batch,  )
                            #callbacks=[early_stopping]) #, check_point])
#         plt.plot(history.history['loss'], label='loss')
#         plt.ylim([0, 1])
#         plt.xlabel('Iteration',fontweight="bold",fontsize = 15)
#         plt.ylabel('Loss',fontweight="bold",fontsize = 15)
#         plt.title("Cost Function",fontweight="bold",fontsize = 20)
#         plt.legend()
#         plt.show()
        predicted = []
        result = model.predict(X_test)
        for i in range(X_test.shape[0]):
            if result[i] <= 0.5:
                predicted.append(0)
            else:
                predicted.append(1)
        get_results(y_test, predicted)
    print("Acc:{}\nF1:{}\nGM:{}\nBA:{}\nPRE:{}\nREC:{}\nSPE:{}\n".format(np.mean(list_acc),np.mean(list_f1),np.mean(list_gmean),
                                                                         np.mean(list_bacc),np.mean(list_pre),np.mean(list_rec),
                                                                         np.mean(list_spe)))     
    mse_acc.append(np.mean(list_acc))
    mse_f1.append(np.mean(list_f1))
    mse_gmean.append(np.mean(list_gmean))
    mse_bacc.append(np.mean(list_bacc))
    mse_pre.append(np.mean(list_pre))
    mse_rec.append(np.mean(list_rec))
    mse_spe.append(np.mean(list_spe))
               
res['MSE'] = [np.mean(mse_acc), np.mean(mse_f1), np.mean(mse_gmean), np.mean(mse_bacc), 
              np.mean(mse_pre), np.mean(mse_rec), np.mean(mse_spe)]
res 

In [ ]:
print("AC:", np.round(np.mean(mse_acc),4),'±',np.round(np.std(mse_acc),4))
print("F1:", np.round(np.mean(mse_f1),4),'±',np.round(np.std(mse_f1),4))
print("GM:", np.round(np.mean(mse_gmean),4),'±',np.round(np.std(mse_gmean),4))
print("BA:", np.round(np.mean(mse_bacc),4),'±',np.round(np.std(mse_bacc),4))
print("PRE:", np.round(np.mean(mse_pre),4),'±',np.round(np.std(mse_pre),4))
print("REC:", np.round(np.mean(mse_rec),4),'±',np.round(np.std(mse_rec),4))
print("SPE:", np.round(np.mean(mse_spe),4),'±',np.round(np.std(mse_spe),4))

# BCE

In [ ]:
bce_acc = []
bce_f1 = []
bce_gmean = []
bce_bacc = []
bce_pre = []
bce_rec = []
bce_spe = []

for i in range(5):
    print('#'*50,'{0}th repeat'.format(i+1),'#'*50)
    list_acc, list_f1, list_gmean, list_bacc, list_pre, list_rec, list_spe = make_lists()

    n_iter=0
    ###################### MLP (sigmoid // MSE) ##############################
    for train_index, test_index in skf.split(X, y):
        n_iter += 1
        X_train = X.iloc[train_index]
        y_train= y.iloc[train_index]
        X_test = X.iloc[test_index]
        y_test= y.iloc[test_index]
        X_train = np.array(X_train)
        y_train = np.array(y_train)
        y_train = y_train.astype(float)
        X_test = np.array(X_test)
        y_test = np.array(y_test)
        y_test = y_test.astype(float)
        batch = int(X_train.shape[0] * 0.05)

        model = create_model()
        opt = tf.keras.optimizers.Adam(learning_rate = 0.01)
        model.compile(optimizer=opt, loss=BCE, metrics=['accuracy'])
        history = model.fit(X_train, y_train, validation_data=(X_test, y_test), verbose=0, epochs=epochs, batch_size = batch,  )
                            #callbacks=[early_stopping]) #, check_point])
#         plt.plot(history.history['loss'], label='loss')
#         plt.ylim([0, 1])
#         plt.xlabel('Iteration',fontweight="bold",fontsize = 15)
#         plt.ylabel('Loss',fontweight="bold",fontsize = 15)
#         plt.title("Cost Function",fontweight="bold",fontsize = 20)
#         plt.legend()
#         plt.show()
        predicted = []
        result = model.predict(X_test)
        for i in range(X_test.shape[0]):
            if result[i] <= 0.5:
                predicted.append(0)
            else:
                predicted.append(1)
        get_results(y_test, predicted)
    print("Acc:{}\nF1:{}\nGM:{}\nBA:{}\nPRE:{}\nREC:{}\nSPE:{}\n".format(np.mean(list_acc),np.mean(list_f1),np.mean(list_gmean),
                                                                         np.mean(list_bacc),np.mean(list_pre),np.mean(list_rec),
                                                                         np.mean(list_spe)))     
    bce_acc.append(np.mean(list_acc))
    bce_f1.append(np.mean(list_f1))
    bce_gmean.append(np.mean(list_gmean))
    bce_bacc.append(np.mean(list_bacc))
    bce_pre.append(np.mean(list_pre))
    bce_rec.append(np.mean(list_rec))
    bce_spe.append(np.mean(list_spe))
           
res['BCE'] = [np.mean(bce_acc), np.mean(bce_f1), np.mean(bce_gmean), np.mean(bce_bacc),
              np.mean(bce_pre), np.mean(bce_rec), np.mean(bce_spe)]
res 

In [ ]:
print("AC:", np.round(np.mean(bce_acc),4),'±',np.round(np.std(bce_acc),4))
print("F1:", np.round(np.mean(bce_f1),4),'±',np.round(np.std(bce_f1),4))
print("GM:", np.round(np.mean(bce_gmean),4),'±',np.round(np.std(bce_gmean),4))
print("BA:", np.round(np.mean(bce_bacc),4),'±',np.round(np.std(bce_bacc),4))
print("PRE:", np.round(np.mean(bce_pre),4),'±',np.round(np.std(bce_pre),4))
print("REC:", np.round(np.mean(bce_rec),4),'±',np.round(np.std(bce_rec),4))
print("SPE:", np.round(np.mean(bce_spe),4),'±',np.round(np.std(bce_spe),4))

# WBCE

In [ ]:
wbce_acc = []
wbce_f1 = []
wbce_gmean = []
wbce_bacc = []
wbce_pre = []
wbce_rec = []
wbce_spe = []

for i in range(5):
    print('#'*50,'{0}th repeat'.format(i+1),'#'*50)
    list_acc, list_f1, list_gmean, list_bacc, list_pre, list_rec, list_spe = make_lists()

    n_iter=0
    ###################### MLP (sigmoid // MSE) ##############################
    for train_index, test_index in skf.split(X, y):
        n_iter += 1
        X_train = X.iloc[train_index]
        y_train= y.iloc[train_index]
        X_test = X.iloc[test_index]
        y_test= y.iloc[test_index]
        X_train = np.array(X_train)
        y_train = np.array(y_train)
        y_train = y_train.astype(float)
        X_test = np.array(X_test)
        y_test = np.array(y_test)
        y_test = y_test.astype(float)
        batch = int(X_train.shape[0] * 0.05)

        model = create_model()
        opt = tf.keras.optimizers.Adam(learning_rate = 0.01)
        model.compile(optimizer=opt, loss=WBCE, metrics=['accuracy'])
        history = model.fit(X_train, y_train, validation_data=(X_test, y_test), verbose=0, epochs=epochs, batch_size = batch,  )
                            #callbacks=[early_stopping]) #, check_point])
#         plt.plot(history.history['loss'], label='loss')
#         plt.ylim([0, 1])
#         plt.xlabel('Iteration',fontweight="bold",fontsize = 15)
#         plt.ylabel('Loss',fontweight="bold",fontsize = 15)
#         plt.title("Cost Function",fontweight="bold",fontsize = 20)
#         plt.legend()
#         plt.show()
        predicted = []
        result = model.predict(X_test)
        for i in range(X_test.shape[0]):
            if result[i] <= 0.5:
                predicted.append(0)
            else:
                predicted.append(1)
        get_results(y_test, predicted)
    print("Acc:{}\nF1:{}\nGM:{}\nBA:{}\nPRE:{}\nREC:{}\nSPE:{}\n".format(np.mean(list_acc),np.mean(list_f1),np.mean(list_gmean),
                                                                         np.mean(list_bacc),np.mean(list_pre),np.mean(list_rec),
                                                                         np.mean(list_spe)))     
    wbce_acc.append(np.mean(list_acc))
    wbce_f1.append(np.mean(list_f1))
    wbce_gmean.append(np.mean(list_gmean))
    wbce_bacc.append(np.mean(list_bacc))
    wbce_pre.append(np.mean(list_pre))
    wbce_rec.append(np.mean(list_rec))
    wbce_spe.append(np.mean(list_spe))
    
res['WBCE'] = [np.mean(wbce_acc), np.mean(wbce_f1), np.mean(wbce_gmean), np.mean(wbce_bacc),
               np.mean(wbce_pre), np.mean(wbce_rec), np.mean(wbce_spe)]
res 

In [ ]:
print("AC:", np.round(np.mean(wbce_acc),4),'±',np.round(np.std(wbce_acc),4))
print("F1:", np.round(np.mean(wbce_f1),4),'±',np.round(np.std(wbce_f1),4))
print("GM:", np.round(np.mean(wbce_gmean),4),'±',np.round(np.std(wbce_gmean),4))
print("BA:", np.round(np.mean(wbce_bacc),4),'±',np.round(np.std(wbce_bacc),4))
print("PRE:", np.round(np.mean(wbce_pre),4),'±',np.round(np.std(wbce_pre),4))
print("REC:", np.round(np.mean(wbce_rec),4),'±',np.round(np.std(wbce_rec),4))
print("SPE:", np.round(np.mean(wbce_spe),4),'±',np.round(np.std(wbce_spe),4))

# Focal

In [ ]:
focal_acc = []
focal_f1 = []
focal_gmean = []
focal_bacc = []
focal_pre = []
focal_rec = []
focal_spe = []

for i in range(5):
    print('#'*50,'{0}th repeat'.format(i+1),'#'*50)
    list_acc, list_f1, list_gmean, list_bacc, list_pre, list_rec, list_spe = make_lists()

    n_iter=0
    ###################### MLP (sigmoid // MSE) ##############################
    for train_index, test_index in skf.split(X, y):
        n_iter += 1
        X_train = X.iloc[train_index]
        y_train= y.iloc[train_index]
        X_test = X.iloc[test_index]
        y_test= y.iloc[test_index]
        X_train = np.array(X_train)
        y_train = np.array(y_train)
        y_train = y_train.astype(float)
        X_test = np.array(X_test)
        y_test = np.array(y_test)
        y_test = y_test.astype(float)
        batch = int(X_train.shape[0] * 0.05)

        model = create_model()
        opt = tf.keras.optimizers.Adam(learning_rate = 0.01)
        model.compile(optimizer=opt, loss=keras.losses.BinaryFocalCrossentropy(), metrics=['accuracy'])
        history = model.fit(X_train, y_train, validation_data=(X_test, y_test), verbose=0, epochs=epochs, batch_size = batch,  )
                            #callbacks=[early_stopping]) #, check_point])
#         plt.plot(history.history['loss'], label='loss')
#         plt.ylim([0, 1])
#         plt.xlabel('Iteration',fontweight="bold",fontsize = 15)
#         plt.ylabel('Loss',fontweight="bold",fontsize = 15)
#         plt.title("Cost Function",fontweight="bold",fontsize = 20)
#         plt.legend()
#         plt.show()
        predicted = []
        result = model.predict(X_test)
        for i in range(X_test.shape[0]):
            if result[i] <= 0.5:
                predicted.append(0)
            else:
                predicted.append(1)
        get_results(y_test, predicted)
    print("Acc:{}\nF1:{}\nGM:{}\nBA:{}\nPRE:{}\nREC:{}\nSPE:{}\n".format(np.mean(list_acc),np.mean(list_f1),np.mean(list_gmean),
                                                                         np.mean(list_bacc),np.mean(list_pre),np.mean(list_rec),
                                                                         np.mean(list_spe)))     
    focal_acc.append(np.mean(list_acc))
    focal_f1.append(np.mean(list_f1))
    focal_gmean.append(np.mean(list_gmean))
    focal_bacc.append(np.mean(list_bacc))
    focal_pre.append(np.mean(list_pre))
    focal_rec.append(np.mean(list_rec))
    focal_spe.append(np.mean(list_spe))
    
res['Focal'] = [np.mean(focal_acc), np.mean(focal_f1), np.mean(focal_gmean), np.mean(focal_bacc),
               np.mean(focal_pre), np.mean(focal_rec), np.mean(focal_spe)]
res 

In [ ]:
print("AC:", np.round(np.mean(focal_acc),4),'±',np.round(np.std(focal_acc),4))
print("F1:", np.round(np.mean(focal_f1),4),'±',np.round(np.std(focal_f1),4))
print("GM:", np.round(np.mean(focal_gmean),4),'±',np.round(np.std(focal_gmean),4))
print("BA:", np.round(np.mean(focal_bacc),4),'±',np.round(np.std(focal_bacc),4))
print("PRE:", np.round(np.mean(focal_pre),4),'±',np.round(np.std(focal_pre),4))
print("REC:", np.round(np.mean(focal_rec),4),'±',np.round(np.std(focal_rec),4))
print("SPE:", np.round(np.mean(focal_spe),4),'±',np.round(np.std(focal_spe),4))

# F1

In [ ]:
L = 10

f1_acc = []
f1_f1 = []
f1_gmean= []
f1_bacc = []
f1_pre = []
f1_rec = []
f1_spe = []

for i in range(5):
    print('#'*50,'{0}th repeat'.format(i+1),'#'*50)
    list_acc, list_f1, list_gmean, list_bacc, list_pre, list_rec, list_spe = make_lists()

    n_iter=0
    ###################### MLP (sigmoid // MSE) ##############################
    for train_index, test_index in skf.split(X, y):
        n_iter += 1
        X_train = X.iloc[train_index]
        y_train= y.iloc[train_index]
        X_test = X.iloc[test_index]
        y_test= y.iloc[test_index]
        X_train = np.array(X_train)
        y_train = np.array(y_train)
        y_train = y_train.astype(float)
        X_test = np.array(X_test)
        y_test = np.array(y_test)
        y_test = y_test.astype(float)
        batch = int(X_train.shape[0] * 0.05)

        model = create_model()
        opt = tf.keras.optimizers.Adam(learning_rate = 0.01)
        model.compile(optimizer=opt, loss=Any_Fbeta, metrics=['accuracy'])
        history = model.fit(X_train, y_train, validation_data=(X_test, y_test), verbose=0, epochs=epochs, batch_size = batch,  )
                            #callbacks=[early_stopping]) #, check_point])
#         plt.plot(history.history['loss'], label='loss')
#         plt.ylim([0, 1])
#         plt.xlabel('Iteration',fontweight="bold",fontsize = 15)
#         plt.ylabel('Loss',fontweight="bold",fontsize = 15)
#         plt.title("Cost Function",fontweight="bold",fontsize = 20)
#         plt.legend()
#         plt.show()
        predicted = []
        result = model.predict(X_test)
        for i in range(X_test.shape[0]):
            if result[i] <= 0.5:
                predicted.append(0)
            else:
                predicted.append(1)
        get_results(y_test, predicted)
    print("Acc:{}\nF1:{}\nGM:{}\nBA:{}\nPRE:{}\nREC:{}\nSPE:{}\n".format(np.mean(list_acc),np.mean(list_f1),np.mean(list_gmean),
                                                                         np.mean(list_bacc),np.mean(list_pre),np.mean(list_rec),
                                                                         np.mean(list_spe)))     
    f1_acc.append(np.mean(list_acc))
    f1_f1.append(np.mean(list_f1))
    f1_gmean.append(np.mean(list_gmean))
    f1_bacc.append(np.mean(list_bacc))
    f1_pre.append(np.mean(list_pre))
    f1_rec.append(np.mean(list_rec))
    f1_spe.append(np.mean(list_spe))
    
res['F1'] = [np.mean(f1_acc), np.mean(f1_f1), np.mean(f1_gmean), np.mean(f1_bacc),
            np.mean(f1_pre), np.mean(f1_rec), np.mean(f1_spe)]
res

In [ ]:
print("AC:", np.round(np.mean(f1_acc),4),'±',np.round(np.std(f1_acc),4))
print("F1:", np.round(np.mean(f1_f1),4),'±',np.round(np.std(f1_f1),4))
print("GM:", np.round(np.mean(f1_gmean),4),'±',np.round(np.std(f1_gmean),4))
print("BA:", np.round(np.mean(f1_bacc),4),'±',np.round(np.std(f1_bacc),4))
print("PRE:", np.round(np.mean(f1_pre),4),'±',np.round(np.std(f1_pre),4))
print("REC:", np.round(np.mean(f1_rec),4),'±',np.round(np.std(f1_rec),4))
print("SPE:", np.round(np.mean(f1_spe),4),'±',np.round(np.std(f1_spe),4))

In [ ]:
r = 0.8   # default value

bcefl_acc = []
bcefl_f1 = []
bcefl_gmean = []
bcefl_bacc = []
bcefl_pre = []
bcefl_rec = []
bcefl_spe = []

for i in range(5):
    print('#'*50,'{0}th repeat'.format(i+1),'#'*50)
    list_acc, list_f1, list_gmean, list_bacc, list_pre, list_rec, list_spe = make_lists()

    n_iter=0
    ###################### MLP (sigmoid // MSE) ##############################
    for train_index, test_index in skf.split(X, y):
        n_iter += 1
        X_train = X.iloc[train_index]
        y_train= y.iloc[train_index]
        X_test = X.iloc[test_index]
        y_test= y.iloc[test_index]
        X_train = np.array(X_train)
        y_train = np.array(y_train)
        y_train = y_train.astype(float)
        X_test = np.array(X_test)
        y_test = np.array(y_test)
        y_test = y_test.astype(float)
        batch = int(X_train.shape[0] * 0.05)

        model = create_model()
        opt = tf.keras.optimizers.Adam(learning_rate = 0.01)
        model.compile(optimizer=opt, loss=WBCEFL, metrics=['accuracy'])
        history = model.fit(X_train, y_train, validation_data=(X_test, y_test), verbose=0, epochs=epochs, batch_size = batch,  )
                            #callbacks=[early_stopping]) #, check_point])
#         plt.plot(history.history['loss'], label='loss')
#         plt.ylim([0, 1])
#         plt.xlabel('Iteration',fontweight="bold",fontsize = 15)
#         plt.ylabel('Loss',fontweight="bold",fontsize = 15)
#         plt.title("Cost Function",fontweight="bold",fontsize = 20)
#         plt.legend()
#         plt.show()
        predicted = []
        result = model.predict(X_test)
        for i in range(X_test.shape[0]):
            if result[i] <= 0.5:
                predicted.append(0)
            else:
                predicted.append(1)
        get_results(y_test, predicted)
    print("Acc:{}\nF1:{}\nGM:{}\nBA:{}\nPRE:{}\nREC:{}\nSPE:{}\n".format(np.mean(list_acc),np.mean(list_f1),np.mean(list_gmean),
                                                                         np.mean(list_bacc),np.mean(list_pre),np.mean(list_rec),
                                                                         np.mean(list_spe)))     
    bcefl_acc.append(np.mean(list_acc))
    bcefl_f1.append(np.mean(list_f1))
    bcefl_gmean.append(np.mean(list_gmean))
    bcefl_bacc.append(np.mean(list_bacc))
    bcefl_pre.append(np.mean(list_pre))
    bcefl_rec.append(np.mean(list_rec))
    bcefl_spe.append(np.mean(list_spe))
    
res['WBCEFL'] = [np.mean(bcefl_acc), np.mean(bcefl_f1), np.mean(bcefl_gmean), np.mean(bcefl_bacc),
               np.mean(bcefl_pre), np.mean(bcefl_rec), np.mean(bcefl_spe)]
res

In [ ]:
print("AC:", np.round(np.mean(bcefl_acc),4),'±',np.round(np.std(bcefl_acc),4))
print("F1:", np.round(np.mean(bcefl_f1),4),'±',np.round(np.std(bcefl_f1),4))
print("GM:", np.round(np.mean(bcefl_gmean),4),'±',np.round(np.std(bcefl_gmean),4))
print("BA:", np.round(np.mean(bcefl_bacc),4),'±',np.round(np.std(bcefl_bacc),4))
print("PRE:", np.round(np.mean(bcefl_pre),4),'±',np.round(np.std(bcefl_pre),4))
print("REC:", np.round(np.mean(bcefl_rec),4),'±',np.round(np.std(bcefl_rec),4))
print("SPE:", np.round(np.mean(bcefl_spe),4),'±',np.round(np.std(bcefl_spe),4))

# GM

In [ ]:
L = 10

gmean_acc = []
gmean_f1 = []
gmean_gmean= []
gmean_bacc = []
gmean_pre = []
gmean_rec = []
gmean_spe = []

for i in range(5):
    print('#'*50,'{0}th repeat'.format(i+1),'#'*50)
    list_acc, list_f1, list_gmean, list_bacc, list_pre, list_rec, list_spe = make_lists()

    n_iter=0
    ###################### MLP (sigmoid // MSE) ##############################
    for train_index, test_index in skf.split(X, y):
        n_iter += 1
        X_train = X.iloc[train_index]
        y_train= y.iloc[train_index]
        X_test = X.iloc[test_index]
        y_test= y.iloc[test_index]
        X_train = np.array(X_train)
        y_train = np.array(y_train)
        y_train = y_train.astype(float)
        X_test = np.array(X_test)
        y_test = np.array(y_test)
        y_test = y_test.astype(float)
        batch = int(X_train.shape[0] * 0.05)

        model = create_model()
        opt = tf.keras.optimizers.Adam(learning_rate = 0.01)
        model.compile(optimizer=opt, loss=Any_Gmean, metrics=['accuracy'])
        history = model.fit(X_train, y_train, validation_data=(X_test, y_test), verbose=0, epochs=epochs, batch_size = batch,  )
                            #callbacks=[early_stopping]) #, check_point])
#         plt.plot(history.history['loss'], label='loss')
#         plt.ylim([0, 1])
#         plt.xlabel('Iteration',fontweight="bold",fontsize = 15)
#         plt.ylabel('Loss',fontweight="bold",fontsize = 15)
#         plt.title("Cost Function",fontweight="bold",fontsize = 20)
#         plt.legend()
#         plt.show()
        predicted = []
        result = model.predict(X_test)
        for i in range(X_test.shape[0]):
            if result[i] <= 0.5:
                predicted.append(0)
            else:
                predicted.append(1)
        get_results(y_test, predicted)
    print("Acc:{}\nF1:{}\nGM:{}\nBA:{}\nPRE:{}\nREC:{}\nSPE:{}\n".format(np.mean(list_acc),np.mean(list_f1),np.mean(list_gmean),
                                                                         np.mean(list_bacc),np.mean(list_pre),np.mean(list_rec),
                                                                         np.mean(list_spe)))     
    gmean_acc.append(np.mean(list_acc))
    gmean_f1.append(np.mean(list_f1))
    gmean_gmean.append(np.mean(list_gmean))
    gmean_bacc.append(np.mean(list_bacc))
    gmean_pre.append(np.mean(list_pre))
    gmean_rec.append(np.mean(list_rec))
    gmean_spe.append(np.mean(list_spe))
    
res['GM'] = [np.mean(gmean_acc), np.mean(gmean_f1), np.mean(gmean_gmean), np.mean(gmean_bacc),
            np.mean(gmean_pre), np.mean(gmean_rec), np.mean(gmean_spe)]
res

In [ ]:
print("AC:", np.round(np.mean(gmean_acc),4),'±',np.round(np.std(gmean_acc),4))
print("F1:", np.round(np.mean(gmean_f1),4),'±',np.round(np.std(gmean_f1),4))
print("GM:", np.round(np.mean(gmean_gmean),4),'±',np.round(np.std(gmean_gmean),4))
print("BA:", np.round(np.mean(gmean_bacc),4),'±',np.round(np.std(gmean_bacc),4))
print("PRE:", np.round(np.mean(gmean_pre),4),'±',np.round(np.std(gmean_pre),4))
print("REC:", np.round(np.mean(gmean_rec),4),'±',np.round(np.std(gmean_rec),4))
print("SPE:", np.round(np.mean(gmean_spe),4),'±',np.round(np.std(gmean_spe),4))

In [ ]:
r = 0.9   # default value

bcegl_acc = []
bcegl_f1 = []
bcegl_gmean = []
bcegl_bacc = []
bcegl_pre = []
bcegl_rec = []
bcegl_spe = []

for i in range(5):
    print('#'*50,'{0}th repeat'.format(i+1),'#'*50)
    list_acc, list_f1, list_gmean, list_bacc, list_pre, list_rec, list_spe = make_lists()

    n_iter=0
    ###################### MLP (sigmoid // MSE) ##############################
    for train_index, test_index in skf.split(X, y):
        n_iter += 1
        X_train = X.iloc[train_index]
        y_train= y.iloc[train_index]
        X_test = X.iloc[test_index]
        y_test= y.iloc[test_index]
        X_train = np.array(X_train)
        y_train = np.array(y_train)
        y_train = y_train.astype(float)
        X_test = np.array(X_test)
        y_test = np.array(y_test)
        y_test = y_test.astype(float)
        batch = int(X_train.shape[0] * 0.05)

        model = create_model()
        opt = tf.keras.optimizers.Adam(learning_rate = 0.01)
        model.compile(optimizer=opt, loss=WBCEGL, metrics=['accuracy'])
        history = model.fit(X_train, y_train, validation_data=(X_test, y_test), verbose=0, epochs=epochs, batch_size = batch,  )
                            #callbacks=[early_stopping]) #, check_point])
#         plt.plot(history.history['loss'], label='loss')
#         plt.ylim([0, 1])
#         plt.xlabel('Iteration',fontweight="bold",fontsize = 15)
#         plt.ylabel('Loss',fontweight="bold",fontsize = 15)
#         plt.title("Cost Function",fontweight="bold",fontsize = 20)
#         plt.legend()
#         plt.show()
        predicted = []
        result = model.predict(X_test)
        for i in range(X_test.shape[0]):
            if result[i] <= 0.5:
                predicted.append(0)
            else:
                predicted.append(1)
        get_results(y_test, predicted)
    print("Acc:{}\nF1:{}\nGM:{}\nBA:{}\nPRE:{}\nREC:{}\nSPE:{}\n".format(np.mean(list_acc),np.mean(list_f1),np.mean(list_gmean),
                                                                         np.mean(list_bacc),np.mean(list_pre),np.mean(list_rec),
                                                                         np.mean(list_spe)))     
    bcegl_acc.append(np.mean(list_acc))
    bcegl_f1.append(np.mean(list_f1))
    bcegl_gmean.append(np.mean(list_gmean))
    bcegl_bacc.append(np.mean(list_bacc))
    bcegl_pre.append(np.mean(list_pre))
    bcegl_rec.append(np.mean(list_rec))
    bcegl_spe.append(np.mean(list_spe))
    
res['WBCEGL'] = [np.mean(bcegl_acc), np.mean(bcegl_f1), np.mean(bcegl_gmean), np.mean(bcegl_bacc),
                np.mean(bcegl_pre), np.mean(bcegl_rec), np.mean(bcegl_spe)]
res

In [ ]:
print("AC:", np.round(np.mean(bcegl_acc),4),'±',np.round(np.std(bcegl_acc),4))
print("F1:", np.round(np.mean(bcegl_f1),4),'±',np.round(np.std(bcegl_f1),4))
print("GM:", np.round(np.mean(bcegl_gmean),4),'±',np.round(np.std(bcegl_gmean),4))
print("BA:", np.round(np.mean(bcegl_bacc),4),'±',np.round(np.std(bcegl_bacc),4))
print("PRE:", np.round(np.mean(bcegl_pre),4),'±',np.round(np.std(bcegl_pre),4))
print("REC:", np.round(np.mean(bcegl_rec),4),'±',np.round(np.std(bcegl_rec),4))
print("SPE:", np.round(np.mean(bcegl_spe),4),'±',np.round(np.std(bcegl_spe),4))

# BA

In [ ]:
L = 10

bacc_acc = []
bacc_f1 = []
bacc_gmean = []
bacc_bacc = []
bacc_pre = []
bacc_rec = []
bacc_spe = []

for i in range(5):
    print('#'*50,'{0}th repeat'.format(i+1),'#'*50)
    list_acc, list_f1, list_gmean, list_bacc, list_pre, list_rec, list_spe = make_lists()

    n_iter=0
    ###################### MLP (sigmoid // MSE) ##############################
    for train_index, test_index in skf.split(X, y):
        n_iter += 1
        X_train = X.iloc[train_index]
        y_train= y.iloc[train_index]
        X_test = X.iloc[test_index]
        y_test= y.iloc[test_index]
        X_train = np.array(X_train)
        y_train = np.array(y_train)
        y_train = y_train.astype(float)
        X_test = np.array(X_test)
        y_test = np.array(y_test)
        y_test = y_test.astype(float)
        batch = int(X_train.shape[0] * 0.05)

        model = create_model()
        opt = tf.keras.optimizers.Adam(learning_rate = 0.01)
        model.compile(optimizer=opt, loss=Any_BAccu, metrics=['accuracy'])
        history = model.fit(X_train, y_train, validation_data=(X_test, y_test), verbose=0, epochs=epochs, batch_size = batch,  )
                            #callbacks=[early_stopping]) #, check_point])
#         plt.plot(history.history['loss'], label='loss')
#         plt.ylim([0, 1])
#         plt.xlabel('Iteration',fontweight="bold",fontsize = 15)
#         plt.ylabel('Loss',fontweight="bold",fontsize = 15)
#         plt.title("Cost Function",fontweight="bold",fontsize = 20)
#         plt.legend()
#         plt.show()
        predicted = []
        result = model.predict(X_test)
        for i in range(X_test.shape[0]):
            if result[i] <= 0.5:
                predicted.append(0)
            else:
                predicted.append(1)
        get_results(y_test, predicted)
    print("Acc:{}\nF1:{}\nGM:{}\nBA:{}\nPRE:{}\nREC:{}\nSPE:{}\n".format(np.mean(list_acc),np.mean(list_f1),np.mean(list_gmean),
                                                                         np.mean(list_bacc),np.mean(list_pre),np.mean(list_rec),
                                                                         np.mean(list_spe)))     
    bacc_acc.append(np.mean(list_acc))
    bacc_f1.append(np.mean(list_f1))
    bacc_gmean.append(np.mean(list_gmean))
    bacc_bacc.append(np.mean(list_bacc))
    bacc_pre.append(np.mean(list_pre))
    bacc_rec.append(np.mean(list_rec))
    bacc_spe.append(np.mean(list_spe))
    
res['BA'] = [np.mean(bacc_acc), np.mean(bacc_f1), np.mean(bacc_gmean), np.mean(bacc_bacc),
            np.mean(bacc_pre), np.mean(bacc_rec), np.mean(bacc_spe)]
res

In [ ]:
print("AC:", np.round(np.mean(bacc_acc),4),'±',np.round(np.std(bacc_acc),4))
print("F1:", np.round(np.mean(bacc_f1),4),'±',np.round(np.std(bacc_f1),4))
print("GM:", np.round(np.mean(bacc_gmean),4),'±',np.round(np.std(bacc_gmean),4))
print("BA:", np.round(np.mean(bacc_bacc),4),'±',np.round(np.std(bacc_bacc),4))
print("PRE:", np.round(np.mean(bacc_pre),4),'±',np.round(np.std(bacc_pre),4))
print("REC:", np.round(np.mean(bacc_rec),4),'±',np.round(np.std(bacc_rec),4))
print("SPE:", np.round(np.mean(bacc_spe),4),'±',np.round(np.std(bacc_spe),4))

In [ ]:
r = 0.9   # default value

bcebl_acc = []
bcebl_f1 = []
bcebl_gmean = []
bcebl_bacc = []
bcebl_pre = []
bcebl_rec = []
bcebl_spe = []

for i in range(5):
    print('#'*50,'{0}th repeat'.format(i+1),'#'*50)
    list_acc, list_f1, list_gmean, list_bacc, list_pre, list_rec, list_spe = make_lists()

    n_iter=0
    ###################### MLP (sigmoid // MSE) ##############################
    for train_index, test_index in skf.split(X, y):
        n_iter += 1
        X_train = X.iloc[train_index]
        y_train= y.iloc[train_index]
        X_test = X.iloc[test_index]
        y_test= y.iloc[test_index]
        X_train = np.array(X_train)
        y_train = np.array(y_train)
        y_train = y_train.astype(float)
        X_test = np.array(X_test)
        y_test = np.array(y_test)
        y_test = y_test.astype(float)
        batch = int(X_train.shape[0] * 0.05)

        model = create_model()
        opt = tf.keras.optimizers.Adam(learning_rate = 0.01)
        model.compile(optimizer=opt, loss=WBCEBL, metrics=['accuracy'])
        history = model.fit(X_train, y_train, validation_data=(X_test, y_test), verbose=0, epochs=epochs, batch_size = batch,  )
                            #callbacks=[early_stopping]) #, check_point])
#         plt.plot(history.history['loss'], label='loss')
#         plt.ylim([0, 1])
#         plt.xlabel('Iteration',fontweight="bold",fontsize = 15)
#         plt.ylabel('Loss',fontweight="bold",fontsize = 15)
#         plt.title("Cost Function",fontweight="bold",fontsize = 20)
#         plt.legend()
#         plt.show()
        predicted = []
        result = model.predict(X_test)
        for i in range(X_test.shape[0]):
            if result[i] <= 0.5:
                predicted.append(0)
            else:
                predicted.append(1)
        get_results(y_test, predicted)
    print("Acc:{}\nF1:{}\nGM:{}\nBA:{}\nPRE:{}\nREC:{}\nSPE:{}\n".format(np.mean(list_acc),np.mean(list_f1),np.mean(list_gmean),
                                                                         np.mean(list_bacc),np.mean(list_pre),np.mean(list_rec),
                                                                         np.mean(list_spe)))     
    bcebl_acc.append(np.mean(list_acc))
    bcebl_f1.append(np.mean(list_f1))
    bcebl_gmean.append(np.mean(list_gmean))
    bcebl_bacc.append(np.mean(list_bacc))
    bcebl_pre.append(np.mean(list_pre))
    bcebl_rec.append(np.mean(list_rec))
    bcebl_spe.append(np.mean(list_spe))
    
res['WBCEBL'] = [np.mean(bcebl_acc), np.mean(bcebl_f1), np.mean(bcebl_gmean), np.mean(bcebl_bacc),
                np.mean(bcebl_pre), np.mean(bcebl_rec), np.mean(bcebl_spe)]
res

In [ ]:
print("AC:", np.round(np.mean(bcebl_acc),4),'±',np.round(np.std(bcebl_acc),4))
print("F1:", np.round(np.mean(bcebl_f1),4),'±',np.round(np.std(bcebl_f1),4))
print("GM:", np.round(np.mean(bcebl_gmean),4),'±',np.round(np.std(bcebl_gmean),4))
print("BA:", np.round(np.mean(bcebl_bacc),4),'±',np.round(np.std(bcebl_bacc),4))
print("PRE:", np.round(np.mean(bcebl_pre),4),'±',np.round(np.std(bcebl_pre),4))
print("REC:", np.round(np.mean(bcebl_rec),4),'±',np.round(np.std(bcebl_rec),4))
print("SPE:", np.round(np.mean(bcebl_spe),4),'±',np.round(np.std(bcebl_spe),4))

In [ ]:
res

# Sensitivity Analysis

### F1

In [ ]:
res = pd.DataFrame({'MSE':[0, 0, 0, 0, 0, 0, 0]}, index = ['Acc','F1','G_Mean','B_Acc','Pre','Rec','Spe'])
res

In [ ]:
R = [0.1, 0.3, 0.5, 0.7, 0.9]
for r in R:
    bcefl_acc = []
    bcefl_f1 = []
    bcefl_gmean = []
    bcefl_bacc = []
    bcefl_pre = []
    bcefl_rec = []
    bcefl_spe = []

    for i in range(5):
        print('#'*50,'{0}th repeat'.format(i+1),'#'*50)
        list_acc, list_f1, list_gmean, list_bacc, list_pre, list_rec, list_spe = make_lists()

        n_iter=0
        ###################### MLP (sigmoid // MSE) ##############################
        for train_index, test_index in skf.split(X, y):
            n_iter += 1
            X_train = X.iloc[train_index]
            y_train= y.iloc[train_index]
            X_test = X.iloc[test_index]
            y_test= y.iloc[test_index]
            X_train = np.array(X_train)
            y_train = np.array(y_train)
            y_train = y_train.astype(float)
            X_test = np.array(X_test)
            y_test = np.array(y_test)
            y_test = y_test.astype(float)
            batch = int(X_train.shape[0] * 0.05)

            model = create_model()
            opt = tf.keras.optimizers.Adam(learning_rate = 0.01)
            model.compile(optimizer=opt, loss=WBCEFL, metrics=['accuracy'])
            history = model.fit(X_train, y_train, validation_data=(X_test, y_test), verbose=0, epochs=epochs, batch_size = batch,  )
                                #callbacks=[early_stopping]) #, check_point])
#             plt.plot(history.history['loss'], label='loss')
#             plt.ylim([0, 1])
#             plt.xlabel('Iteration',fontweight="bold",fontsize = 15)
#             plt.ylabel('Loss',fontweight="bold",fontsize = 15)
#             plt.title("Cost Function",fontweight="bold",fontsize = 20)
#             plt.legend()
#             plt.show()
            predicted = []
            result = model.predict(X_test)
            for i in range(X_test.shape[0]):
                if result[i] <= 0.5:
                    predicted.append(0)
                else:
                    predicted.append(1)
            get_results(y_test, predicted)
        print("Acc:{}\nF1:{}\nGM:{}\nBA:{}\nPRE:{}\nREC:{}\nSPE:{}\n".format(np.mean(list_acc),np.mean(list_f1),np.mean(list_gmean),
                                                                             np.mean(list_bacc),np.mean(list_pre),np.mean(list_rec),
                                                                             np.mean(list_spe)))     
        bcefl_acc.append(np.mean(list_acc))
        bcefl_f1.append(np.mean(list_f1))
        bcefl_gmean.append(np.mean(list_gmean))
        bcefl_bacc.append(np.mean(list_bacc))
        bcefl_pre.append(np.mean(list_pre))
        bcefl_rec.append(np.mean(list_rec))
        bcefl_spe.append(np.mean(list_spe))

    res['WBCEFL_{}'.format(r)] = [np.mean(bcefl_acc), np.mean(bcefl_f1), np.mean(bcefl_gmean), np.mean(bcefl_bacc),
                                  np.mean(bcefl_pre), np.mean(bcefl_rec), np.mean(bcefl_spe)]
res

### GM

In [ ]:
R = [0.1, 0.3, 0.5, 0.7, 0.9]
for r in R:
    bcegl_acc = []
    bcegl_f1 = []
    bcegl_gmean = []
    bcegl_bacc = []
    bcegl_pre = []
    bcegl_rec = []
    bcegl_spe = []

    for i in range(5):
        print('#'*50,'{0}th repeat'.format(i+1),'#'*50)
        list_acc, list_f1, list_gmean, list_bacc, list_pre, list_rec, list_spe = make_lists()

        n_iter=0
        ###################### MLP (sigmoid // MSE) ##############################
        for train_index, test_index in skf.split(X, y):
            n_iter += 1
            X_train = X.iloc[train_index]
            y_train= y.iloc[train_index]
            X_test = X.iloc[test_index]
            y_test= y.iloc[test_index]
            X_train = np.array(X_train)
            y_train = np.array(y_train)
            y_train = y_train.astype(float)
            X_test = np.array(X_test)
            y_test = np.array(y_test)
            y_test = y_test.astype(float)
            batch = int(X_train.shape[0] * 0.05)

            model = create_model()
            opt = tf.keras.optimizers.Adam(learning_rate = 0.01)
            model.compile(optimizer=opt, loss=WBCEGL, metrics=['accuracy'])
            history = model.fit(X_train, y_train, validation_data=(X_test, y_test), verbose=0, epochs=epochs, batch_size = batch,  )
                                #callbacks=[early_stopping]) #, check_point])
#             plt.plot(history.history['loss'], label='loss')
#             plt.ylim([0, 1])
#             plt.xlabel('Iteration',fontweight="bold",fontsize = 15)
#             plt.ylabel('Loss',fontweight="bold",fontsize = 15)
#             plt.title("Cost Function",fontweight="bold",fontsize = 20)
#             plt.legend()
#             plt.show()
            predicted = []
            result = model.predict(X_test)
            for i in range(X_test.shape[0]):
                if result[i] <= 0.5:
                    predicted.append(0)
                else:
                    predicted.append(1)
            get_results(y_test, predicted)
        print("Acc:{}\nF1:{}\nGM:{}\nBA:{}\nPRE:{}\nREC:{}\nSPE:{}\n".format(np.mean(list_acc),np.mean(list_f1),np.mean(list_gmean),
                                                                             np.mean(list_bacc),np.mean(list_pre),np.mean(list_rec),
                                                                             np.mean(list_spe)))     
        bcegl_acc.append(np.mean(list_acc))
        bcegl_f1.append(np.mean(list_f1))
        bcegl_gmean.append(np.mean(list_gmean))
        bcegl_bacc.append(np.mean(list_bacc))
        bcegl_pre.append(np.mean(list_pre))
        bcegl_rec.append(np.mean(list_rec))
        bcegl_spe.append(np.mean(list_spe))

    res['WBCEGL_{}'.format(r)] = [np.mean(bcegl_acc), np.mean(bcegl_f1), np.mean(bcegl_gmean), np.mean(bcegl_bacc),
                                  np.mean(bcegl_pre), np.mean(bcegl_rec), np.mean(bcegl_spe)]
res

### BA

In [ ]:
R = [0.1, 0.3, 0.5, 0.7, 0.9]
for r in R:
    bcebl_acc = []
    bcebl_f1 = []
    bcebl_gmean = []
    bcebl_bacc = []
    bcebl_pre = []
    bcebl_rec = []
    bcebl_spe = []

    for i in range(5):
        print('#'*50,'{0}th repeat'.format(i+1),'#'*50)
        list_acc, list_f1, list_gmean, list_bacc, list_pre, list_rec, list_spe = make_lists()

        n_iter=0
        ###################### MLP (sigmoid // MSE) ##############################
        for train_index, test_index in skf.split(X, y):
            n_iter += 1
            X_train = X.iloc[train_index]
            y_train= y.iloc[train_index]
            X_test = X.iloc[test_index]
            y_test= y.iloc[test_index]
            X_train = np.array(X_train)
            y_train = np.array(y_train)
            y_train = y_train.astype(float)
            X_test = np.array(X_test)
            y_test = np.array(y_test)
            y_test = y_test.astype(float)
            batch = int(X_train.shape[0] * 0.05)

            model = create_model()
            opt = tf.keras.optimizers.Adam(learning_rate = 0.01)
            model.compile(optimizer=opt, loss=WBCEBL, metrics=['accuracy'])
            history = model.fit(X_train, y_train, validation_data=(X_test, y_test), verbose=0, epochs=epochs, batch_size = batch,  )
                                #callbacks=[early_stopping]) #, check_point])
#             plt.plot(history.history['loss'], label='loss')
#             plt.ylim([0, 1])
#             plt.xlabel('Iteration',fontweight="bold",fontsize = 15)
#             plt.ylabel('Loss',fontweight="bold",fontsize = 15)
#             plt.title("Cost Function",fontweight="bold",fontsize = 20)
#             plt.legend()
#             plt.show()
            predicted = []
            result = model.predict(X_test)
            for i in range(X_test.shape[0]):
                if result[i] <= 0.5:
                    predicted.append(0)
                else:
                    predicted.append(1)
            get_results(y_test, predicted)
        print("Acc:{}\nF1:{}\nGM:{}\nBA:{}\nPRE:{}\nREC:{}\nSPE:{}\n".format(np.mean(list_acc),np.mean(list_f1),np.mean(list_gmean),
                                                                             np.mean(list_bacc),np.mean(list_pre),np.mean(list_rec),
                                                                             np.mean(list_spe)))     
        bcebl_acc.append(np.mean(list_acc))
        bcebl_f1.append(np.mean(list_f1))
        bcebl_gmean.append(np.mean(list_gmean))
        bcebl_bacc.append(np.mean(list_bacc))
        bcebl_pre.append(np.mean(list_pre))
        bcebl_rec.append(np.mean(list_rec))
        bcebl_spe.append(np.mean(list_spe))

    res['WBCEBL_{}'.format(r)] = [np.mean(bcebl_acc), np.mean(bcebl_f1), np.mean(bcebl_gmean), np.mean(bcebl_bacc),
                                  np.mean(bcebl_pre), np.mean(bcebl_rec), np.mean(bcebl_spe)]
res

In [ ]:
res

In [ ]:
res_f1 = list(res[res.index == 'F1'].iloc[0,1:6])
res_f1

In [ ]:
res_gm = list(res[res.index == 'G_Mean'].iloc[0,6:11])
res_gm

In [ ]:
res_ba = list(res[res.index == 'B_Acc'].iloc[0,11:16])
res_ba

In [ ]:
data = np.array([res_f1, res_gm, res_ba])
data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Labels
columns = ['0.1', '0.3', '0.5', '0.7', '0.9']   # X-axis
rows = ['F1', 'GM', 'BA']                      # Legend

# X positions
x = np.arange(len(columns))

plt.figure(figsize=(8, 6))

# Plot each row as a separate line
for i, metric in enumerate(rows):
    plt.plot(x, data[i, :], marker='o', label=metric, linewidth=2)

# X-axis setup
plt.xticks(x, columns)
plt.xlabel('Hyperparameter Value')
plt.ylabel('Score')
plt.title('Performance vs Hyperparameter')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
